# Homework 5 — Responsible AI in Practice
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · Posted Week 12 · **Due: end of Week 14, Sunday 11:59 pm (Arizona time)** on Brightspace · 100 points · Late work: see the syllabus late-work table (two grace days per semester, then −10% per day, up to 3 days).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/homework/hw5-responsible-ai-in-practice/HW5_Responsible_AI_in_Practice.ipynb)

Three tasks that a junior engineer at a firm using AI could face in their first year: **catch the errors** in an AI-written calculation before it goes out the door, **audit** a vendor's AI model before your team relies on it, and **advise** your firm on whether to adopt an AI tool at all.

### What you will be able to do
1. Apply the 5-check verification workflow (Restate, Units & magnitude, Independent check, Sources, Test & document) to find and fix errors in an AI-generated engineering calculation.
2. Audit a machine-learning model for distribution shift and subgroup error, and explain why the *direction* of an error matters for safety.
3. Write a one-page recommendation using the NIST AI RMF (Govern, Map, Measure, Manage) and engineering ethics (public safety, competence, responsible charge).

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `concrete_strength.csv` (CC BY 4.0) and `osha/osha_1926_subpartP_excavations.txt` (U.S. Government work) from the course `data/` folder.

**AI-use label:** Part A is **AI-Free** (no AI tools; Python, notes, and the OSHA text are fine). **Turn off Colab's AI code suggestions before Part A** — see *Turning off AI suggestions* in the course [Colab setup guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/colab-setup.md). Part B is **AI-Assisted**. Part C is **AI-Assisted with a required acknowledgment**. Fill in the AI-use acknowledgment at the end for Parts B and C; a missing acknowledgment costs **5 points**.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

---
## Part A — Find the errors (AI-Free · 35 points)
A coworker asked a chat assistant to check a storm drain design and pasted the answer below into the project
file. It reads well. **It contains four planted errors.** Some steps are correct — fixing a correct step earns no
credit, so check before you change anything.

Use the 5 checks from Lab 12: **(1) Restate, (2) Units & magnitude, (3) Independent check, (4) Sources,
(5) Test & document.**

> ### 🤖 AI-generated calculation (pasted from a chat assistant — do not trust it yet)
> **Project:** New storm drain for an asphalt parking lot, Tucson, AZ.
> **Given:** drainage area A = 3.5 acres, all asphalt pavement · design rainfall intensity i = 3.2 in/hr ·
> proposed pipe: 18-in diameter reinforced concrete pipe, Manning's n = 0.013, slope 0.5% ·
> trench depth 10 ft, trench bottom width 4 ft, soil classified as **Type C** by the competent person ·
> construction easement 30 ft wide.
>
> **Step 1 — Peak runoff (rational method).** Q = C·i·A, with Q in cfs (cubic feet per second), i in in/hr, and A in acres
> (1 acre·in/hr ≈ 1.008 cfs, so no conversion factor is needed). For a parking lot, a runoff coefficient of
> C = 0.35 is typical. Q = 0.35 × 3.2 × 3.5 = **3.92 cfs**.
>
> **Step 2 — Pipe geometry.** D = 18 in = 1.5 ft. Flow area A_p = πD²/4 = 1.767 ft². For a circular pipe
> flowing full, the hydraulic radius is R = D/2 = 0.75 ft.
>
> **Step 3 — Full-flow capacity (Manning's equation, U.S. units).** V = (1.49/n)·R^(2/3)·S^(1/2).
> With S = 0.5% = 0.05: V = (1.49/0.013)(0.75)^(2/3)(0.05)^(1/2) = 114.6 × 0.8255 × 0.2236 = **21.2 ft/s**.
> Q_full = V·A_p = 21.2 × 1.767 = **37.4 cfs**.
>
> **Step 4 — Trench excavation.** Per OSHA 29 CFR 1926.652(b)(5), trenches in Type C soil that will be open
> less than 24 hours may be sloped at 1H:1V (45°). Top width = 4 + 2(1 × 10) = **24 ft**.
>
> **Conclusion.** Capacity 37.4 cfs ≫ demand 3.92 cfs (ratio 9.5). The 18-in pipe is adequate with a large
> margin, and the velocity of 21.2 ft/s ensures excellent self-cleansing. A 12-in pipe could be considered to
> reduce cost. The 24-ft trench fits within the 30-ft construction easement.

**✍️ Question:** **A1. Restate.** In your own words: what is given (with units), what is being asked, and what decision depends on the answer? (3–5 sentences) *(4 pts)*

*Your answer:* 

**A2. The four errors (16 points, 4 per error).** Double-click this cell and fill in one row per
error. *Check that caught it* = which of the 5 checks (more than one is fine).

| # | Step | What is wrong | Check that caught it | Correct value or statement |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |

**A3. Independent check (8 points).** Redo the calculation correctly in Python. State your runoff
coefficient as an assumption. Then check the next pipe sizes up (21 in and 24 in).

> 💡 **Hint:** 0.5% = 0.005. For a full circular pipe R = area / wetted perimeter = D/4. Type C = 1½ horizontal per 1 vertical on each side.

In [ ]:
# TODO: corrected calculation. Keep units in the comments.
C = None            # runoff coefficient for asphalt (your assumption)
i = 3.2             # in/hr
A = 3.5             # acres
Q_peak = None       # cfs

def full_flow_capacity(D_in, n=0.013, S=None):   # TODO: replace None with the slope in ft/ft (0.5% = ?)
    """Full-flow capacity (cfs) of a circular pipe of diameter D_in (inches), Manning's equation in U.S. units."""
    D = D_in / 12               # ft
    area = None                 # ft²
    R = None                    # ft
    V = None                    # ft/s
    return V * area

for D_in in [18, 21, 24]:
    print(f"{D_in}-in pipe: capacity {full_flow_capacity(D_in):.2f} cfs vs demand {Q_peak:.2f} cfs")

top_width = None    # ft, Type C slope from OSHA Table B-1
print(f"Trench top width: {top_width:.0f} ft")

In [ ]:
# Sources check (Check 4): search the regulation text yourself.
osha = open(get_data("osha/osha_1926_subpartP_excavations.txt"), encoding="utf-8").read()
print("Is there a paragraph (b)(5) in § 1926.652?", "(b)(5)" in osha)
for line in osha.splitlines():
    if line.startswith(("Type C |", "Note 2:")):
        print(line)

**✍️ Question:** **A4. Corrected conclusion (4 points).** Rewrite the conclusion in 2–3 sentences: is the 18-in pipe adequate, what would you recommend, and what does the corrected trench width mean for the 30-ft easement? *(4 pts)*

*Your answer:* 

**✍️ Question:** **A5. Sources and documentation (3 points).** Name the source you would cite for (a) the runoff coefficient and (b) the excavation slope. Then write one sentence for the project file documenting what you checked. *(3 pts)*

*Your answer:* 

---
## Part B — Audit a vendor's model (AI-Assisted · 35 points)
A software vendor sells a model that predicts concrete compressive strength (MPa) from a mix design and age.
Their brochure says: *"Mean absolute error about 3 MPa on held-out test data."* Your firm is considering it for
early-age strength estimates (e.g., when to strip formwork) on **fly-ash mixes**, which your suppliers use a lot.

Your job: **Measure** whether the claim holds for *your* use. The vendor's training data did not include any
fly-ash mixes — you will find out what that means.

The next cell builds the vendor's model exactly as delivered. **Do not change it.**

In [ ]:
# ---- Vendor model (provided as-is; do not edit) ----
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

concrete = pd.read_csv(get_data("concrete_strength.csv"))
inputs = [c for c in concrete.columns if c != "strength_mpa"]

vendor_data = concrete[concrete["fly_ash_kg_m3"] == 0]          # the vendor only had mixes without fly ash
fly_ash_mixes = concrete[concrete["fly_ash_kg_m3"] > 0].copy()  # your firm's typical mixes
vendor_train, vendor_test = train_test_split(vendor_data, test_size=0.25, random_state=42)
vendor_model = RandomForestRegressor(n_estimators=200, random_state=42)
vendor_model.fit(vendor_train[inputs], vendor_train["strength_mpa"])
print(f"Vendor training mixes: {len(vendor_train)}, vendor test mixes: {len(vendor_test)}, "
      f"your fly-ash mixes: {len(fly_ash_mixes)}")

**B1 (5 points).** Reproduce the vendor's claim: compute the MAE (MPa) on the vendor's own test set.

> 💡 **Hint:** `vendor_model.predict(vendor_test[inputs])`, then `mean_absolute_error(true, predicted)`.

In [ ]:
# TODO B1
vendor_test = vendor_test.copy()
vendor_test["pred"] = None
mae_vendor = None
print(f"Vendor test MAE: {mae_vendor:.2f} MPa")

**B2 (8 points).** Now test on your fly-ash mixes. Besides MAE, compute the **mean bias** = average of
(predicted − actual). A positive bias means the model **over-predicts** strength — the *unconservative*
direction: you might strip forms or load a member before the concrete is actually strong enough. Also compute the
share of mixes over-predicted by more than 5 MPa.

> 💡 **Hint:** `error = pred - strength_mpa`. The mean of `error.abs()` is the MAE; the mean of `error` is the bias; `(error > 5).mean()` is a share because True counts as 1.

In [ ]:
# TODO B2
fly_ash_mixes["pred"] = None
fly_ash_mixes["error"] = None          # predicted minus actual, MPa
mae_fly = None
bias_fly = None
share_over5 = None                     # fraction of mixes with error > 5 MPa
print(f"Fly-ash MAE: {mae_fly:.2f} MPa | mean bias: {bias_fly:+.2f} MPa | over-predicted by > 5 MPa: {share_over5:.0%}")

**B3 (8 points).** Subgroup analysis: is the error the same at every age? Group the fly-ash mixes by
test age and report count, MAE, mean bias, and share over-predicted by > 5 MPa for each group. Then make a bar
chart of the mean bias by age group (title, axis labels with units).

> 💡 **Hint:** `groupby("age_group", observed=True)["error"].agg(n="count", MAE=lambda e: e.abs().mean(), bias="mean", share_over5=lambda e: (e > 5).mean())`.

In [ ]:
# TODO B3
age_bins = [0, 7, 28, 90, 365]
age_labels = ["1-7 days", "14-28 days", "56-90 days", "91-365 days"]
fly_ash_mixes["age_group"] = pd.cut(fly_ash_mixes["age_days"], age_bins, labels=age_labels)

by_age = None    # one row per age group: n, MAE, bias, share_over5
print(by_age)

# bar chart of by_age["bias"] here

**B4 (6 points).** Plot predicted vs. actual strength for the vendor test set and the fly-ash mixes on one chart, with a dashed 1:1 line, a legend, a title, and axis labels with units.

> 💡 **Hint:** Two `plt.scatter(actual, predicted, ...)` calls with different colors and labels, then `plt.plot([0, 85], [0, 85], "k--")`.

In [ ]:
# TODO B4: predicted vs actual scatter

**✍️ Question:** **B5. Audit conclusion (8 points, 100–150 words).** Is the vendor's "≈3 MPa" claim true for your firm's use? Use your numbers from B1–B4. Explain *why* the model fails where it does (think about what the training data lacked and how fly ash affects strength gain). Then recommend Manage actions: where may it be used, if at all, and what must change first? *(8 pts)*

*Your answer:* 

---
## Part C — Should our firm adopt it? (AI-Assisted with acknowledgment · 25 points)
You work at a 40-person civil/architectural engineering firm in Arizona. A principal asks you for a **one-page
memo (250–350 words)**: *"Should our firm adopt AI tool X for task Y?"* Choose **one** scenario:

1. **Bridge inspection triage:** a drone-photo crack-detection tool that ranks which of a county's 400 bridges
   get a hands-on inspection first (think Lab 13).
2. **Calculation drafting:** a general chat assistant that drafts stormwater calculation packages for permit
   submittals (think Part A and Lab 12).
3. **Early-age concrete strength:** the vendor model from Part B, to decide formwork stripping times.
4. **Concept-design energy:** an AI tool that predicts heating/cooling loads to choose building massing at
   concept design (think Lab 13 Part 2).

**Your memo must include:**
- **Recommendation** in the first two sentences: adopt / adopt with conditions / do not adopt (yet).
- **Map:** the task, the users, who is affected if it fails, and the worst realistic failure.
- **Measure:** what evidence exists and what tests you would require before use (cite your own results from this
  course where relevant).
- **Manage and Govern:** concrete safeguards (human review, thresholds, range checks, logging, monitoring), who
  is accountable, training, and how the firm documents AI use.
- **Professional ethics:** public safety held paramount, practicing within competence, and **responsible charge**
  — the licensed engineer who seals the work remains responsible for it, whatever tools were used.
- **Data:** which class of data the tool would see (Public / Internal / Confidential (client) / Restricted) and
  whether that is allowed.
- Hedge any statement about current laws or products ("as of 2026; check the current version").

Short is fine: one or two sentences per bullet. Then add your **AI-use acknowledgment** for Part C in the final
cell (tool and version, specific use, prompts, how you checked the output).

**✍️ Question:** **Your memo** (To / From / Date / Subject, then the memo). *(25 pts)*

*Your answer:* 

---
## Professionalism (5 points)
Notebook runs top to bottom (**Runtime → Restart session and run all**) and every plot has a title and axis labels
with units. **AI-use acknowledgment:** the final cell must cover Parts B and C (Part A must be AI-Free). A missing
acknowledgment costs 5 points of the 100, as on every homework.

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
1. **Runtime → Restart session and run all** — make sure everything runs without errors (cells that load a model take a few extra minutes).
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.